# Note 07 — Relationship / Deeper Analysis

## 1. Research Question

Những thay đổi nào trong electricity mix có mối liên hệ mạnh nhất với sự thay đổi của carbon intensity, và các mối quan hệ này có ổn định khi xét đến quy mô hệ thống điện hay không?

## 2. Objective

- Phân tích mối quan hệ giữa energy transition và carbon intensity.
- Đánh giá vai trò tương đối của Fossil, Renewables và các renewable sources.
- Kiểm tra liệu kết quả có bị chi phối bởi quy mô hệ thống điện hay không.
- Xác định các transition patterns có mối liên hệ rõ nhất với carbon outcome.
- Kiểm tra độ ổn định của các kết quả trước khi chuyển sang phân tích Carbon Outcome ở Note 08.

## 3. Approach

Sử dụng các transition features từ Note 06 để phân tích relationship giữa thay đổi electricity mix và carbon intensity, sau đó thực hiện robustness checks theo system size và so sánh mức độ liên hệ giữa các nguồn điện.

## 4. Expected Outcome

Xác định được những transition variables có mối liên hệ mạnh nhất với carbon intensity và nhận diện các pattern đáng chú ý để tiếp tục kiểm tra trực tiếp bằng carbon generation/emissions ở Note 08.

## 5. Link to Next Note

Các relationship được xác định ở Note 07 sẽ được kiểm tra trực tiếp với carbon-related outcomes trong **Note 08 — Carbon, Capacity & Transition Analysis**.

In [1]:
# ============================================================
# NOTE 7 — RELATIONSHIP SCREENING
#
# RQ6:
# - Những biến số nào có mối quan hệ đáng chú ý với quá trình chuyển dịch cơ cấu điện?
#
# - Các chỉ số thay đổi tỷ trọng nguồn điện có quan hệ như thế nào với nhau trong giai đoạn 2012–2022?
#
# Mục đích: Sàng lọc các mối quan hệ đáng chú ý trước khi chọn một quan hệ để phân tích sâu hơn.
# ============================================================

import pandas as pd
from pathlib import Path

relative_path = Path("checkpoints/note06_country_transition_profiles.pkl")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Không tìm thấy file: {relative_path}"
    )
df_path = project_root / relative_path

checkpoint_path = (
    project_root /
    "checkpoints/note06_country_transition_profiles.pkl"
)

note06=pd.read_pickle(checkpoint_path)
profile = note06["profile"].copy()

print("=" * 70)
print("NOTE 7 — LOAD NOTE 6 CHECKPOINT")
print("=" * 70)
print(f"Loaded: {checkpoint_path}")
print(f"Số quốc gia: {len(profile)}")

# ------------------------------------------------------------
# 7.2: Các biến chuyển dịch dùng cho screening
# ------------------------------------------------------------

relationship_cols = [
    "Delta_Fossil_share_pp",
    "Delta_Renewables_share_pp",
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp",
    "Delta_Nuclear_share_pp",
    "Transition_intensity_pp",
    "Generation_2022_TWh"
]

corr_matrix = (
    profile[relationship_cols]
    .corr(method="pearson")
    .round(3)
)

print("\n" + "=" * 70)
print("MA TRẬN TƯƠNG QUAN PEARSON")
print("=" * 70)

print(corr_matrix)

print("\n" + "=" * 70)
print("CÁC CẶP CÓ |r| >= 0.50")
print("=" * 70)

corr_pairs = []

for i in range(len(relationship_cols)):
    for j in range(i + 1, len(relationship_cols)):
        var1 = relationship_cols[i]
        var2 = relationship_cols[j]
        r = corr_matrix.loc[var1, var2]

        if abs(r) >= 0.50:
            corr_pairs.append({
                "Variable_1": var1,
                "Variable_2": var2,
                "Correlation": r
            })

corr_pairs_df = (
    pd.DataFrame(corr_pairs)
    .sort_values(
        "Correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
    .reset_index(drop=True)
)

print(corr_pairs_df)

NOTE 7 — LOAD NOTE 6 CHECKPOINT
Loaded: d:\phan tich data\Ember_Electricity\checkpoints\note06_country_transition_profiles.pkl
Số quốc gia: 209

MA TRẬN TƯƠNG QUAN PEARSON
                           Delta_Fossil_share_pp  Delta_Renewables_share_pp  \
Delta_Fossil_share_pp                      1.000                     -0.986   
Delta_Renewables_share_pp                 -0.986                      1.000   
Delta_Solar_share_pp                      -0.524                      0.528   
Delta_Wind_share_pp                       -0.564                      0.587   
Delta_Hydro_share_pp                      -0.536                      0.534   
Delta_Nuclear_share_pp                    -0.079                     -0.078   
Transition_intensity_pp                   -0.492                      0.481   
Generation_2022_TWh                       -0.038                      0.033   

                           Delta_Solar_share_pp  Delta_Wind_share_pp  \
Delta_Fossil_share_pp                    -0.

In [2]:
# ============================================================
# 7.3: ROBUSTNESS CHECK
#
# Quy mô hệ thống điện có thực sự liên quan đến cường độ
# chuyển dịch hay kết quả Pearson bị ảnh hưởng bởi phân phối Generation rất lệch?
#
# Mục đích: So sánh Pearson và Spearman giữa Generation_2022_TWh và Transition_intensity_pp.
# ============================================================

from scipy.stats import spearmanr

# ------------------------------------------------------------
# Pearson đã có từ Step 7.2
# ------------------------------------------------------------

pearson_r = profile[
    [
        "Generation_2022_TWh",
        "Transition_intensity_pp"
    ]
].corr().loc[
    "Generation_2022_TWh",
    "Transition_intensity_pp"
]

# ------------------------------------------------------------
# Spearman: phù hợp hơn để kiểm tra quan hệ đơn điệu khi biến Generation có phân phối lệch mạnh.
# ------------------------------------------------------------

spearman_r, spearman_p = spearmanr(
    profile["Generation_2022_TWh"],
    profile["Transition_intensity_pp"],
    nan_policy="omit"
)

print("=" * 70)
print("QUAN HỆ GIỮA QUY MÔ HỆ THỐNG VÀ CƯỜNG ĐỘ CHUYỂN DỊCH")
print("=" * 70)

print(f"Pearson r  = {pearson_r:.3f}")
print(f"Spearman ρ = {spearman_r:.3f}")
print(f"Spearman p = {spearman_p:.4f}")

print("\n" + "=" * 70)
print("DIỄN GIẢI")
print("=" * 70)

if abs(spearman_r) < 0.10:
    print(
        "Spearman |ρ| < 0.10 → gần như không có quan hệ đơn điệu "
        "giữa quy mô hệ thống và cường độ chuyển dịch."
    )
elif abs(spearman_r) < 0.30:
    print(
        "Spearman |ρ| < 0.30 → quan hệ đơn điệu yếu."
    )
elif abs(spearman_r) < 0.50:
    print(
        "Spearman |ρ| < 0.50 → quan hệ đơn điệu ở mức vừa."
    )
else:
    print(
        "Spearman |ρ| >= 0.50 → có quan hệ đơn điệu tương đối mạnh."
    )

QUAN HỆ GIỮA QUY MÔ HỆ THỐNG VÀ CƯỜNG ĐỘ CHUYỂN DỊCH
Pearson r  = -0.009
Spearman ρ = 0.034
Spearman p = 0.6246

DIỄN GIẢI
Spearman |ρ| < 0.10 → gần như không có quan hệ đơn điệu giữa quy mô hệ thống và cường độ chuyển dịch.


In [3]:
# ============================================================
# 7.4: TRANSITION INTENSITY × RENEWABLE DRIVERS
#
# Cường độ giảm fossil có liên hệ như thế nào với mức tăng Solar, Wind và Hydro?
#
# Mục đích: So sánh Pearson và Spearman để xác định nguồn tái tạo nào có quan hệ rõ nhất với cường độ chuyển dịch.
# ============================================================

from scipy.stats import spearmanr

driver_cols = [
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp"
]

results = []

for col in driver_cols:

    valid = profile[
        ["Transition_intensity_pp", col]
    ].dropna()

    pearson_r = valid[
        ["Transition_intensity_pp", col]
    ].corr().iloc[0, 1]

    spearman_r, spearman_p = spearmanr(
        valid["Transition_intensity_pp"],
        valid[col]
    )

    results.append({
        "Renewable_driver": col,
        "N": len(valid),
        "Pearson_r": pearson_r,
        "Spearman_rho": spearman_r,
        "Spearman_p": spearman_p
    })

driver_relationships = (
    pd.DataFrame(results)
    .sort_values(
        "Spearman_rho",
        key=lambda x: x.abs(),
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("TRANSITION INTENSITY × RENEWABLE DRIVERS")
print("=" * 70)

print(
    driver_relationships.round(3)
)

print("\n" + "=" * 70)
print("XẾP HẠNG THEO |SPEARMAN ρ|")
print("=" * 70)

for i, row in driver_relationships.iterrows():
    print(
        f"{i + 1}. {row['Renewable_driver']}: "
        f"ρ = {row['Spearman_rho']:.3f}, "
        f"p = {row['Spearman_p']:.4f}"
    )

TRANSITION INTENSITY × RENEWABLE DRIVERS
       Renewable_driver    N  Pearson_r  Spearman_rho  Spearman_p
0  Delta_Solar_share_pp  209      0.468         0.451       0.000
1   Delta_Wind_share_pp  201      0.230         0.273       0.000
2  Delta_Hydro_share_pp  205      0.050         0.100       0.155

XẾP HẠNG THEO |SPEARMAN ρ|
1. Delta_Solar_share_pp: ρ = 0.451, p = 0.0000
2. Delta_Wind_share_pp: ρ = 0.273, p = 0.0001
3. Delta_Hydro_share_pp: ρ = 0.100, p = 0.1548


In [4]:
# ============================================================
# 7.5: DRIVERS WITHIN FOSSIL DECLINE
#
# Trong các quốc gia thực sự giảm tỷ trọng fossil, mức độ chuyển dịch có liên hệ rõ hơn với Solar, Wind hay Hydro?
#
# Mục đích: Loại bỏ nhóm Fossil growth và Stable/moderate để kiểm tra
# xem kết quả ở Step 7.4 có còn giữ nguyên trong nhóm chuyển dịch thực sự hay không.
# ============================================================

from scipy.stats import spearmanr

decline = profile[
    profile["Transition_direction"] == "Fossil decline"
].copy()

driver_cols = [
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp"
]

results_decline = []

for col in driver_cols:

    valid = decline[
        ["Transition_intensity_pp", col]
    ].dropna()

    pearson_r = valid[
        ["Transition_intensity_pp", col]
    ].corr().iloc[0, 1]

    spearman_r, spearman_p = spearmanr(
        valid["Transition_intensity_pp"],
        valid[col]
    )

    results_decline.append({
        "Renewable_driver": col,
        "N": len(valid),
        "Pearson_r": pearson_r,
        "Spearman_rho": spearman_r,
        "Spearman_p": spearman_p
    })

decline_driver_relationships = (
    pd.DataFrame(results_decline)
    .sort_values(
        "Spearman_rho",
        key=lambda x: x.abs(),
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("FOSSIL DECLINE — RENEWABLE DRIVERS")
print("=" * 70)

print(f"Số quốc gia được phân tích: {len(decline)}")

print("\n" + "=" * 70)
print("TƯƠNG QUAN VỚI CƯỜNG ĐỘ CHUYỂN DỊCH")
print("=" * 70)

print(
    decline_driver_relationships.round(3)
)

print("\n" + "=" * 70)
print("XẾP HẠNG THEO |SPEARMAN ρ|")
print("=" * 70)

for i, row in decline_driver_relationships.iterrows():
    print(
        f"{i + 1}. {row['Renewable_driver']}: "
        f"ρ = {row['Spearman_rho']:.3f}, "
        f"p = {row['Spearman_p']:.4f}"
    )

FOSSIL DECLINE — RENEWABLE DRIVERS
Số quốc gia được phân tích: 94

TƯƠNG QUAN VỚI CƯỜNG ĐỘ CHUYỂN DỊCH
       Renewable_driver   N  Pearson_r  Spearman_rho  Spearman_p
0   Delta_Wind_share_pp  90      0.439         0.300       0.004
1  Delta_Hydro_share_pp  92      0.245         0.277       0.008
2  Delta_Solar_share_pp  94      0.517         0.245       0.018

XẾP HẠNG THEO |SPEARMAN ρ|
1. Delta_Wind_share_pp: ρ = 0.300, p = 0.0041
2. Delta_Hydro_share_pp: ρ = 0.277, p = 0.0076
3. Delta_Solar_share_pp: ρ = 0.245, p = 0.0175


In [5]:
# ============================================================
# 7.6: EXTREME TRANSITION CASES
#
# Chênh lệch lớn giữa Pearson và Spearman có xuất phát từ
# một số quốc gia có mức tăng Solar/Wind/Hydro cực đoan hay không?
#
# Mục đích:
# Kiểm tra các trường hợp có giá trị driver lớn nhất và xem
# chúng đóng góp như thế nào vào cường độ chuyển dịch.
# ============================================================

# ------------------------------------------------------------
# Kiểm tra các trường hợp cực đoan của từng renewable driver
# ------------------------------------------------------------

driver_cols = [
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp"
]

for col in driver_cols:

    print("\n" + "=" * 70)
    print(f"TOP 10 — {col}")
    print("=" * 70)

    cols_show = [
        "Area",
        "Transition_intensity_pp",
        "Delta_Fossil_share_pp",
        "Delta_Renewables_share_pp",
        col
    ]

    print(
        decline[cols_show]
        .dropna(subset=[col])
        .sort_values(col, ascending=False)
        .head(10)
        .round(3)
    )

# ------------------------------------------------------------
# Các trường hợp có cường độ chuyển dịch lớn nhất
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 15 — CƯỜNG ĐỘ CHUYỂN DỊCH CAO NHẤT")
print("=" * 70)

top_intensity = [
    "Area",
    "Transition_intensity_pp",
    "Delta_Fossil_share_pp",
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp"
]

print(
    decline[top_intensity]
    .sort_values(
        "Transition_intensity_pp",
        ascending=False
    )
    .head(15)
    .round(3)
)


TOP 10 — Delta_Solar_share_pp
                     Area  Transition_intensity_pp  Delta_Fossil_share_pp  \
42     Cook Islands (the)                   50.000                -50.000   
143  Palestine (State of)                   33.333                -33.333   
105               Lebanon                   39.463                -39.463   
99               Kiribati                   25.000                -25.000   
110            Luxembourg                   74.718                -74.718   
55            El Salvador                   30.164                -30.164   
36                  Chile                   19.047                -19.047   
206                 Yemen                   16.832                -16.832   
171               Somalia                   19.048                -19.048   
96                 Jordan                   22.706                -22.706   

     Delta_Renewables_share_pp  Delta_Solar_share_pp  
42                      50.000                50.000  
143        

Solar có Pearson cao nhất trong nhóm Fossil decline nhưng chênh lệch đáng kể giữa Pearson và Spearman cho thấy mối quan hệ tuyến tính này có thể chịu ảnh hưởng của một số trường hợp cực đoan. Wind có Spearman cao nhất, cho thấy mức tăng Wind có quan hệ đơn điệu rõ hơn với cường độ chuyển dịch trong nhóm các quốc gia giảm tỷ trọng fossil. Hydro có tương quan yếu hơn nhưng vẫn có ý nghĩa thống kê.

In [6]:
# ============================================================
# 7.7: ROBUSTNESS CHECK
#
# Sau khi giảm ảnh hưởng của các trường hợp chuyển dịch cực đoan,
# mối quan hệ giữa Transition intensity và Solar/Wind/Hydro có còn giữ nguyên không?
#
# Mục đích: Kiểm tra robustness của kết quả Step 7.5 và xác định xem
# thứ hạng Solar / Wind / Hydro có phụ thuộc quá nhiều vào outlier.
# ============================================================

from scipy.stats import pearsonr, spearmanr

# Chỉ xét nhóm quốc gia có Fossil decline
robust_results = []

for driver in [
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp"
]:
    
    temp = decline[
        ["Area", "Transition_intensity_pp", driver]
    ].dropna().copy()
    
    # Loại các trường hợp cực đoan theo percentile 95%
    upper = temp[driver].quantile(0.95)
    temp_robust = temp[temp[driver] <= upper].copy()
    
    x = temp_robust["Transition_intensity_pp"]
    y = temp_robust[driver]
    
    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)
    
    robust_results.append({
        "Driver": driver,
        "N_original": len(temp),
        "N_robust": len(temp_robust),
        "Removed": len(temp) - len(temp_robust),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p,
        "95pct_cutoff": upper
    })

robust_results_df = (
    pd.DataFrame(robust_results)
    .sort_values(
        "Spearman_rho",
        ascending=False
    )
)

print("=" * 80)
print("ROBUSTNESS CHECK")
print("=" * 80)

print(
    robust_results_df.round(4).to_string(index=False)
)

print("\n" + "=" * 80)
print("THỨ HẠNG THEO SPEARMAN SAU KHI LOẠI EXTREME CASES")
print("=" * 80)

print(
    robust_results_df[
        ["Driver", "Spearman_rho", "Spearman_p", "N_robust"]
    ]
    .round(4)
    .to_string(index=False)
)

ROBUSTNESS CHECK
              Driver  N_original  N_robust  Removed  Pearson_r  Pearson_p  Spearman_rho  Spearman_p  95pct_cutoff
Delta_Hydro_share_pp          92        87        5     0.1268     0.2419        0.1854      0.0856       16.8047
 Delta_Wind_share_pp          90        85        5     0.0779     0.4784        0.1829      0.0939       19.9425
Delta_Solar_share_pp          94        89        5     0.1702     0.1108        0.1208      0.2594       19.7488

THỨ HẠNG THEO SPEARMAN SAU KHI LOẠI EXTREME CASES
              Driver  Spearman_rho  Spearman_p  N_robust
Delta_Hydro_share_pp        0.1854      0.0856        87
 Delta_Wind_share_pp        0.1829      0.0939        85
Delta_Solar_share_pp        0.1208      0.2594        89


Trong nhóm quốc gia có tỷ trọng fossil giảm, Wind, Hydro và Solar đều cho thấy tương quan dương với cường độ chuyển dịch trong phân tích ban đầu. Tuy nhiên, khi loại bỏ các trường hợp cực đoan, các hệ số tương quan giảm đáng kể và không còn đạt ý nghĩa thống kê ở mức 5%. Điều này cho thấy mối quan hệ quan sát được trong phân tích ban đầu chịu ảnh hưởng đáng kể từ một số trường hợp chuyển dịch cực đoan.

In [7]:
# ============================================================
# 7.8: OVERALL RELATIONSHIP RANKING
#
# Trên toàn bộ các quốc gia, biến nào có mối quan hệ đáng chú ý nhất với cường độ chuyển dịch?
#
# Mục đích: Xếp hạng các biến theo Pearson và Spearman để xác định relationship nổi bật ở cấp độ toàn bộ dataset.
# ============================================================

from scipy.stats import pearsonr, spearmanr

target = "Transition_intensity_pp"

drivers = [
    "Delta_Fossil_share_pp",
    "Delta_Renewables_share_pp",
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp",
    "Delta_Nuclear_share_pp",
    "Generation_2022_TWh"
]

overall_results = []

for col in drivers:
    temp = profile[[target, col]].dropna()

    pearson_r, pearson_p = pearsonr(
        temp[target],
        temp[col]
    )

    spearman_rho, spearman_p = spearmanr(
        temp[target],
        temp[col]
    )

    overall_results.append({
        "Variable": col,
        "N": len(temp),
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "Spearman_rho": spearman_rho,
        "Spearman_p": spearman_p
    })

overall_results_df = pd.DataFrame(overall_results)

print("=" * 80)
print("OVERALL RELATIONSHIP RANKING")
print("=" * 80)

print(
    overall_results_df
    .sort_values(
        "Spearman_rho",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("XẾP HẠNG THEO |SPEARMAN|")
print("=" * 80)

print(
    overall_results_df[
        ["Variable", "Spearman_rho", "Spearman_p", "N"]
    ]
    .assign(
        Abs_Spearman=lambda x: x["Spearman_rho"].abs()
    )
    .sort_values(
        "Abs_Spearman",
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)

OVERALL RELATIONSHIP RANKING
                 Variable   N  Pearson_r  Pearson_p  Spearman_rho  Spearman_p
    Delta_Fossil_share_pp 209    -0.4923     0.0000       -0.6440      0.0000
Delta_Renewables_share_pp 209     0.4806     0.0000        0.6190      0.0000
     Delta_Solar_share_pp 209     0.4681     0.0000        0.4513      0.0000
      Delta_Wind_share_pp 201     0.2302     0.0010        0.2734      0.0001
     Delta_Hydro_share_pp 205     0.0499     0.4777        0.0997      0.1548
   Delta_Nuclear_share_pp 193     0.0624     0.3887        0.0355      0.6242
      Generation_2022_TWh 209    -0.0087     0.9005        0.0340      0.6246

XẾP HẠNG THEO |SPEARMAN|
                 Variable  Spearman_rho  Spearman_p   N  Abs_Spearman
    Delta_Fossil_share_pp       -0.6440      0.0000 209        0.6440
Delta_Renewables_share_pp        0.6190      0.0000 209        0.6190
     Delta_Solar_share_pp        0.4513      0.0000 209        0.4513
      Delta_Wind_share_pp        0.2734  

In [8]:
# ============================================================
# 7.9: FINAL RELATIONSHIP SUMMARY
#
# Những relationship nào thực sự đáng giữ lại trong kết luận của phân tích?
#
# Mục đích: Tổng hợp kết quả thành một bảng cuối cùng, đồng thời đánh dấu các relationship 
# mang tính định nghĩa/compositional và các relationship có giá trị diễn giải hơn.
# ============================================================

final_relationship = overall_results_df.copy()

final_relationship["Relationship_strength"] = (
    final_relationship["Spearman_rho"].abs()
)

# Phân loại để tránh diễn giải sai các biến cấu thành intensity
final_relationship["Interpretation"] = "Secondary / weak"

final_relationship.loc[
    final_relationship["Variable"].isin([
        "Delta_Fossil_share_pp",
        "Delta_Renewables_share_pp"
    ]),
    "Interpretation"
] = "Structural / compositional"

final_relationship.loc[
    final_relationship["Variable"].isin([
        "Delta_Solar_share_pp",
        "Delta_Wind_share_pp"
    ]),
    "Interpretation"
] = "Potentially informative"

final_relationship.loc[
    final_relationship["Variable"].isin([
        "Delta_Hydro_share_pp",
        "Delta_Nuclear_share_pp",
        "Generation_2022_TWh"
    ]),
    "Interpretation"
] = "No notable relationship"

final_relationship = (
    final_relationship
    .sort_values(
        "Relationship_strength",
        ascending=False
    )
)

print("=" * 90)
print("NOTE 07 — FINAL RELATIONSHIP SUMMARY")
print("=" * 90)

print(
    final_relationship[
        [
            "Variable",
            "N",
            "Pearson_r",
            "Spearman_rho",
            "Spearman_p",
            "Relationship_strength",
            "Interpretation"
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\n" + "=" * 90)
print("KEY FINDINGS")
print("=" * 90)

print(
    "1. Fossil và Renewables có relationship mạnh nhưng mang tính "
    "structural/compositional."
)

print(
    "2. Solar có association mạnh nhất trong nhóm các renewable "
    "drivers có khả năng diễn giải."
)

print(
    "3. Wind có association yếu hơn Solar nhưng vẫn có ý nghĩa "
    "thống kê trên toàn bộ dataset."
)

print(
    "4. Hydro, Nuclear và quy mô Generation 2022 không cho thấy "
    "relationship đáng kể với transition intensity."
)

print(
    "5. Robustness check cho thấy association Solar/Wind/Hydro "
    "giảm đáng kể khi loại extreme cases."
)

NOTE 07 — FINAL RELATIONSHIP SUMMARY
                 Variable   N  Pearson_r  Spearman_rho  Spearman_p  Relationship_strength             Interpretation
    Delta_Fossil_share_pp 209    -0.4923       -0.6440      0.0000                 0.6440 Structural / compositional
Delta_Renewables_share_pp 209     0.4806        0.6190      0.0000                 0.6190 Structural / compositional
     Delta_Solar_share_pp 209     0.4681        0.4513      0.0000                 0.4513    Potentially informative
      Delta_Wind_share_pp 201     0.2302        0.2734      0.0001                 0.2734    Potentially informative
     Delta_Hydro_share_pp 205     0.0499        0.0997      0.1548                 0.0997    No notable relationship
   Delta_Nuclear_share_pp 193     0.0624        0.0355      0.6242                 0.0355    No notable relationship
      Generation_2022_TWh 209    -0.0087        0.0340      0.6246                 0.0340    No notable relationship

KEY FINDINGS
1. Fossil và 

In [11]:
# ============================================================
# SAVE CHECKPOINT - RQ6: Những biến số nào có mối quan hệ đáng chú ý với quá trình chuyển dịch cơ cấu điện?
# ============================================================

from pathlib import Path
import pickle

note06_relative = Path(
    "checkpoints/note06_country_transition_profiles.pkl"
)

project_root = next(
    (
        path.resolve()
        for path in (Path.cwd(), *Path.cwd().resolve().parents)
        if (path / note06_relative).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy checkpoint Note 06 để xác định project_root."
    )

# ------------------------------------------------------------
# 2. TẠO ĐƯỜNG DẪN CHECKPOINT NOTE 07
# ------------------------------------------------------------

checkpoint_dir = project_root / "checkpoints"

checkpoint_path = (
    checkpoint_dir /
    "note07_relationship_deeper_analysis.pkl"
)

print(f"Checkpoint directory: {checkpoint_dir}")

# ------------------------------------------------------------
# 3. GOM CÁC OBJECT ĐÃ PHÂN TÍCH Ở NOTE 07
# ------------------------------------------------------------
profile = note06["profile"].copy()
transition_features = note06["transition_features"].copy()
note07_checkpoint_data = {
    "profile": profile,
    "decline": decline,
    "transition_features": transition_features,
    "overall_results_df": overall_results_df,
    "robust_results_df": robust_results_df,
    "final_relationship": final_relationship,
}

# ------------------------------------------------------------
# 4. THỰC SỰ GHI FILE .PKL
# ------------------------------------------------------------

with open(checkpoint_path, "wb") as f:
    pickle.dump(note07_checkpoint_data, f)

# ------------------------------------------------------------
# 5. KIỂM TRA FILE
# ------------------------------------------------------------

if checkpoint_path.exists():
    print(f"File size: {checkpoint_path.stat().st_size:,} bytes")
    print(f"Absolute path: {checkpoint_path.resolve()}")

    print("\nCác object đã lưu:")

    for key, value in note07_checkpoint_data.items():
        if hasattr(value, "shape"):
            print(f"- {key}: {value.shape}")
        else:
            print(f"- {key}: {type(value).__name__}")

    print("\n✓ CHECKPOINT NOTE 07 ĐÃ ĐƯỢC LƯU THÀNH CÔNG.")
else:
    raise FileNotFoundError(
        f"Không thể xác nhận file checkpoint: {checkpoint_path}"
    )

Checkpoint directory: D:\phan tich data\Ember_Electricity\checkpoints
File size: 61,086 bytes
Absolute path: D:\phan tich data\Ember_Electricity\checkpoints\note07_relationship_deeper_analysis.pkl

Các object đã lưu:
- profile: (209, 13)
- decline: (94, 13)
- transition_features: (209, 7)
- overall_results_df: (7, 6)
- robust_results_df: (3, 9)
- final_relationship: (7, 8)

✓ CHECKPOINT NOTE 07 ĐÃ ĐƯỢC LƯU THÀNH CÔNG.


## Summary

Note 07 cho thấy một số thành phần của electricity mix có relationship rõ ràng với carbon intensity.

Note 08 sẽ kiểm tra sâu hơn liệu energy transition có đi kèm với thay đổi thực tế của generation, emissions và các carbon outcomes hay không.